# End-to-end 2D thrust-fault prediction and post-processing

This notebook is the compact, reproducible version of `fault_process_weak_gradient_weak_curvature_then_angle_continue - Copy.ipynb` for inline 600. It generates the continuous fault-probability map from the SEG-Y input and the trained UASS-Net weights, then applies the complete legacy-compatible 2D post-processing workflow.

Required inputs:

- `model_real.pth` in the repository root. The reference checkpoint SHA-256 is `d154ab68869cfc9c789b94835cb2614c182c7dd0fa3fd56cf2af4bb9b2b638aa`.
- `inline600.segy` at `data/segy/inline600.segy` relative to the repository root. The reference SEG-Y SHA-256 is `87ed66c6c91839661bf4a7a765175cbd332e0922ce85d63c09b44b91539ff785`.

Launch the notebook with the kernel working directory set to either the repository root or its `demo/` directory. The first code cell discovers the repository root without relying on the clone directory name.

For exact numerical compatibility with the source notebook, the decoder uses nearest-neighbor upsampling and the overlap mask preserves the original row-then-column boundary assignment. The continuous sigmoid prediction is passed directly to post-processing; it is not transposed or binarized before the 0.1 probability threshold is applied.

The downstream workflow preserves the source notebook's weak ridge-support recovery, two-pass Group reduction, curvature-constrained weak adjustment, three-point angle filters, second-DBSCAN noise masking with carried labels, iterative geometric merging, post-merge size filter, and final interpolation. Only figure-producing cells retain notebook output.


In [ ]:
from collections import defaultdict
from pathlib import Path
import colorsys
import hashlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import segyio
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.interpolate import interp1d
from scipy.ndimage import (
    binary_dilation,
    distance_transform_edt,
    map_coordinates,
    uniform_filter1d,
)
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN


def find_repository_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    current = current.expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "fault_post_processing_2d.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Launch this notebook from the repository "
        "root or its demo directory."
    )


REPO_ROOT = find_repository_root()
MODEL_WEIGHTS_FILE = REPO_ROOT / "model_real.pth"
SEISMIC_FILE = REPO_ROOT / "data" / "segy" / "inline600.segy"

REFERENCE_MODEL_SHA256 = "d154ab68869cfc9c789b94835cb2614c182c7dd0fa3fd56cf2af4bb9b2b638aa"
REFERENCE_SEISMIC_SHA256 = "87ed66c6c91839661bf4a7a765175cbd332e0922ce85d63c09b44b91539ff785"

INLINE_NUMBER = 600
CROSSLINE_STRIDE = 2
SAMPLE_STRIDE = 2
EXPECTED_PROBABILITY_SHAPE = (2080, 1001)

MODEL_BASE_CHANNELS = 16
MODEL_DROPOUT = 0.2
PREDICTION_PATCH_SHAPE = (256, 256)
PREDICTION_OVERLAP = 12
PREDICTION_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROBABILITY_THRESHOLD = 0.1
GRADIENT_RATIO = 0.10
NORMAL_STEP = 1.0
RIDGE_SUPPORT_RADIUS = 8
MIN_KEEP_RATIO = 0.999
GROUP_MAX_GAP = 20

DENOISE_EPS = 10
DENOISE_MIN_SAMPLES = 5

CURVATURE_TAU = 0.5
CURVATURE_DELTA = 1e-3
CURVATURE_LOCAL_WINDOW = 7
CURVATURE_SMOOTH_WINDOW = 5
CURVATURE_BLEND_STRENGTH = 0.01
CURVATURE_MAX_SHIFT = 0.05
ANGLE_THRESHOLD = 90.0
ANGLE_MAX_ROUNDS = 10

SECOND_DBSCAN_EPS = 10
SECOND_DBSCAN_MIN_SAMPLES = 3

MERGE_THETA_MAX = 17.0
MERGE_DISTANCE_MAX = 75.0
MERGE_EXTRAPOLATION_MAX = 8.0

MIN_CLUSTER_POINTS = 30
MIN_CLUSTER_VERTICAL_RANGE = 30

PLOT_X_LIMITS = (2130, -50)
PLOT_Y_LIMITS = (1051, -50)


## Plotting utilities


In [ ]:
def get_distinct_colors(n_colors, seed=42):
    color_pool = []
    for name in ("tab20", "tab20b", "tab20c", "Set3", "Paired", "Accent", "Dark2"):
        colormap = plt.get_cmap(name)
        if hasattr(colormap, "colors"):
            color_pool.extend(list(colormap.colors))
        else:
            color_pool.extend([colormap(i) for i in np.linspace(0, 1, colormap.N)])

    unique_pool = []
    seen = set()
    for color in color_pool:
        key = tuple(np.round(color, 6))
        if key not in seen:
            seen.add(key)
            unique_pool.append(color)

    rng = np.random.default_rng(seed)
    rng.shuffle(unique_pool)
    if n_colors > len(unique_pool):
        extra = n_colors - len(unique_pool)
        unique_pool.extend([plt.cm.hsv(i / extra) for i in range(extra)])
    return unique_pool[:n_colors]


def format_fault_axis(axis, title=None):
    axis.set_xlim(*PLOT_X_LIMITS)
    axis.set_ylim(*PLOT_Y_LIMITS)
    axis.tick_params(labelsize=11)
    if title:
        axis.set_title(title, fontsize=12)


def draw_points(axis, points, title=None, color="black", size=4):
    points = np.asarray(points)
    if len(points):
        axis.scatter(points[:, 0], points[:, 1], c=color, s=size, marker=".", alpha=0.8)
    format_fault_axis(axis, title)


def draw_clusters(axis, points, labels, title=None, seed=42):
    points = np.asarray(points)
    labels = np.asarray(labels)
    unique_labels = sorted(np.unique(labels).tolist())
    colors = get_distinct_colors(len(unique_labels), seed=seed)

    for cluster_label, color in zip(unique_labels, colors):
        cluster_points = points[labels == cluster_label]
        if len(cluster_points) == 0:
            continue
        axis.plot(
            cluster_points[:, 0],
            cluster_points[:, 1],
            "o",
            markerfacecolor=color,
            markeredgecolor="black",
            markeredgewidth=0.15,
            markersize=3,
            alpha=0.95,
        )
    format_fault_axis(axis, title)


## 1. Generate the inline-600 fault-probability map with UASS-Net

The inference architecture is defined below to keep this notebook self-contained. The supplied file is a raw PyTorch state dictionary and is loaded with strict key and shape validation. Nearest-neighbor decoder upsampling is required to reproduce the source notebook's probability map.

The SEG-Y volume contains inlines 600–602. Inline 600 is selected by its header value, and the crossline and sample axes are each downsampled by two. The resulting `(2080, 1001)` section is predicted with 256 × 256 patches, 12-pixel overlap, patch-wise standardization, sigmoid activation, and Gaussian overlap blending.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, inputs):
        return self.layers(inputs)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden_channels = max(channels // reduction, 1)
        self.layers = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden_channels, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, inputs):
        return inputs * self.layers(inputs)


class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.b1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.b2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=6, dilation=6, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=12, dilation=12, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.b4 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=18, dilation=18, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * 4, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, inputs):
        features = torch.cat(
            [
                self.b1(inputs),
                self.b2(inputs),
                self.b3(inputs),
                self.b4(inputs),
            ],
            dim=1,
        )
        return self.fuse(features)


class DecoderBlock(nn.Module):
    def __init__(self, up_channels, skip_channels, out_channels, dropout):
        super().__init__()
        self.se = SEBlock(skip_channels)
        self.dropout = nn.Dropout2d(dropout)
        self.conv = ConvBlock(up_channels + skip_channels, out_channels)

    def forward(self, inputs, skip):
        # Nearest-neighbor interpolation is part of the reference inference definition.
        inputs = F.interpolate(inputs, size=skip.shape[-2:], mode="nearest")
        skip = self.se(skip)
        inputs = torch.cat([skip, inputs], dim=1)
        inputs = self.dropout(inputs)
        return self.conv(inputs)


class UASSNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=16, dropout=0.2):
        super().__init__()
        channels1 = base_channels
        channels2 = base_channels * 2
        channels3 = base_channels * 4
        channels4 = base_channels * 8
        channels5 = base_channels * 32

        self.enc1 = ConvBlock(in_channels, channels1)
        self.enc2 = ConvBlock(channels1, channels2)
        self.enc3 = ConvBlock(channels2, channels3)
        self.enc4 = ConvBlock(channels3, channels4)
        self.enc5 = ConvBlock(channels4, channels5)
        self.pool = nn.MaxPool2d(2)
        self.aspp = ASPP(channels5, channels4)
        self.dec4 = DecoderBlock(channels4, channels4, channels4, dropout)
        self.dec3 = DecoderBlock(channels4, channels3, channels3, dropout)
        self.dec2 = DecoderBlock(channels3, channels2, channels2, dropout)
        self.dec1 = DecoderBlock(channels2, channels1, channels1, dropout)
        self.out_conv = nn.Conv2d(channels1, out_channels, 3, padding=1)

    def forward(self, inputs):
        encoder1 = self.enc1(inputs)
        encoder2 = self.enc2(self.pool(encoder1))
        encoder3 = self.enc3(self.pool(encoder2))
        encoder4 = self.enc4(self.pool(encoder3))
        encoder5 = self.enc5(self.pool(encoder4))

        decoded = self.aspp(encoder5)
        decoded = self.dec4(decoded, encoder4)
        decoded = self.dec3(decoded, encoder3)
        decoded = self.dec2(decoded, encoder2)
        decoded = self.dec1(decoded, encoder1)
        return self.out_conv(decoded)


In [ ]:
def sha256_file(file_path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as input_file:
        while block := input_file.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def load_inline_section(file_path, inline_number, crossline_stride=2, sample_stride=2):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise FileNotFoundError(f"Seismic input not found: {file_path.resolve()}")

    with segyio.open(str(file_path), "r", strict=False) as seismic_file:
        inline_values = np.asarray(seismic_file.ilines)
        xline_values = np.asarray(seismic_file.xlines)
        sample_count = seismic_file.samples.size
        expected_trace_count = len(inline_values) * len(xline_values)
        if seismic_file.tracecount != expected_trace_count:
            raise ValueError(
                f"SEG-Y trace count {seismic_file.tracecount} does not match "
                f"the regular grid size {expected_trace_count}."
            )

        seismic_traces = np.empty(
            (seismic_file.tracecount, sample_count),
            dtype=np.float32,
        )
        for trace_index, trace in enumerate(seismic_file.trace):
            seismic_traces[trace_index] = trace

    inline_matches = np.flatnonzero(inline_values == inline_number)
    if len(inline_matches) != 1:
        raise ValueError(
            f"Expected one inline {inline_number}, found {len(inline_matches)}."
        )

    seismic_cube = seismic_traces.reshape(
        len(inline_values),
        len(xline_values),
        sample_count,
    )
    section = seismic_cube[
        int(inline_matches[0]),
        ::crossline_stride,
        ::sample_stride,
    ]
    return np.asarray(section, dtype=np.float32)


def minmax_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    image_min = float(np.min(image))
    image_max = float(np.max(image))
    if not image_max > image_min:
        raise ValueError("The seismic section has a constant amplitude.")
    return (image - image_min) / (image_max - image_min)


if not MODEL_WEIGHTS_FILE.is_file():
    raise FileNotFoundError(f"Model weights not found: {MODEL_WEIGHTS_FILE.resolve()}")
if not SEISMIC_FILE.is_file():
    raise FileNotFoundError(f"Seismic input not found: {SEISMIC_FILE.resolve()}")
if sha256_file(MODEL_WEIGHTS_FILE) != REFERENCE_MODEL_SHA256:
    raise ValueError("The model checkpoint does not match the reference SHA-256.")
if sha256_file(SEISMIC_FILE) != REFERENCE_SEISMIC_SHA256:
    raise ValueError("The SEG-Y input does not match the reference SHA-256.")

model = UASSNet(
    in_channels=1,
    out_channels=1,
    base_channels=MODEL_BASE_CHANNELS,
    dropout=MODEL_DROPOUT,
).to(PREDICTION_DEVICE)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="TypedStorage is deprecated")
    try:
        loaded_weights = torch.load(
            MODEL_WEIGHTS_FILE,
            map_location=PREDICTION_DEVICE,
            weights_only=True,
        )
    except TypeError:
        loaded_weights = torch.load(
            MODEL_WEIGHTS_FILE,
            map_location=PREDICTION_DEVICE,
        )

if "model" in loaded_weights:
    loaded_weights = loaded_weights["model"]
elif "model_state_dict" in loaded_weights:
    loaded_weights = loaded_weights["model_state_dict"]

model.load_state_dict(loaded_weights, strict=True)
model.eval()

seismic_section = load_inline_section(
    SEISMIC_FILE,
    inline_number=INLINE_NUMBER,
    crossline_stride=CROSSLINE_STRIDE,
    sample_stride=SAMPLE_STRIDE,
)
seismic_normalized = minmax_normalize(seismic_section)
if seismic_normalized.shape != EXPECTED_PROBABILITY_SHAPE:
    raise ValueError(
        f"Unexpected inline shape {seismic_normalized.shape}; "
        f"expected {EXPECTED_PROBABILITY_SHAPE}."
    )


In [ ]:
def build_gaussian_overlap_weights(overlap, patch_shape):
    patch_rows, patch_columns = patch_shape
    if overlap <= 0 or overlap >= min(patch_shape):
        raise ValueError("Overlap must be positive and smaller than each patch dimension.")

    weights = np.ones(patch_shape, dtype=np.float32)
    sigma_factor = 0.5 / ((overlap / 4.0) ** 2)
    edge_weights = np.asarray(
        [
            np.exp(-((index - overlap + 1) ** 2) * sigma_factor)
            for index in range(overlap)
        ],
        dtype=np.float32,
    )

    # Preserve the source assignment order: rows first, columns second.
    for index in range(overlap):
        weights[index, :] = edge_weights[index]
        weights[patch_rows - index - 1, :] = edge_weights[index]
    for index in range(overlap):
        weights[:, index] = edge_weights[index]
        weights[:, patch_columns - index - 1] = edge_weights[index]
    return weights


def predict_probability_map(model, seismic_image, patch_shape, overlap, device):
    seismic_image = np.asarray(seismic_image, dtype=np.float32)
    if seismic_image.ndim != 2:
        raise ValueError(f"Expected a 2D seismic image, received {seismic_image.shape}.")

    patch_rows, patch_columns = patch_shape
    stride_rows = patch_rows - overlap
    stride_columns = patch_columns - overlap
    image_rows, image_columns = seismic_image.shape

    tile_rows = int(np.round((image_rows + overlap) / stride_rows + 0.5))
    tile_columns = int(np.round((image_columns + overlap) / stride_columns + 0.5))
    padded_rows = stride_rows * tile_rows + overlap
    padded_columns = stride_columns * tile_columns + overlap

    padded_image = np.zeros((padded_rows, padded_columns), dtype=np.float32)
    probability_sum = np.zeros_like(padded_image)
    weight_sum = np.zeros_like(padded_image)
    padded_image[:image_rows, :image_columns] = seismic_image
    overlap_weights = build_gaussian_overlap_weights(overlap, patch_shape)

    with torch.inference_mode():
        for tile_row in range(tile_rows):
            for tile_column in range(tile_columns):
                row_start = tile_row * stride_rows
                column_start = tile_column * stride_columns
                patch = padded_image[
                    row_start : row_start + patch_rows,
                    column_start : column_start + patch_columns,
                ]

                patch_mean = float(np.mean(patch))
                patch_std = float(np.std(patch))
                if not patch_std > 0.0:
                    raise ValueError(
                        f"Constant seismic patch at tile ({tile_row}, {tile_column})."
                    )
                standardized_patch = (patch - patch_mean) / patch_std
                model_input = torch.from_numpy(
                    standardized_patch[None, None]
                ).to(device=device, dtype=torch.float32)
                patch_probability = torch.sigmoid(model(model_input))
                patch_probability = patch_probability[0, 0].cpu().numpy()

                probability_sum[
                    row_start : row_start + patch_rows,
                    column_start : column_start + patch_columns,
                ] += patch_probability * overlap_weights
                weight_sum[
                    row_start : row_start + patch_rows,
                    column_start : column_start + patch_columns,
                ] += overlap_weights

    blended_probability = np.divide(
        probability_sum,
        weight_sum,
        out=np.zeros_like(probability_sum),
        where=weight_sum > 0.0,
    )
    return np.asarray(
        blended_probability[:image_rows, :image_columns],
        dtype=np.float32,
    )


In [ ]:
probability_map = predict_probability_map(
    model=model,
    seismic_image=seismic_normalized,
    patch_shape=PREDICTION_PATCH_SHAPE,
    overlap=PREDICTION_OVERLAP,
    device=PREDICTION_DEVICE,
)

if probability_map.shape != EXPECTED_PROBABILITY_SHAPE:
    raise ValueError(
        f"Unexpected probability-map shape {probability_map.shape}; "
        f"expected {EXPECTED_PROBABILITY_SHAPE}."
    )
if not np.isfinite(probability_map).all():
    raise ValueError("The model prediction contains non-finite values.")
if float(probability_map.min()) < 0.0 or float(probability_map.max()) > 1.0:
    raise ValueError("The sigmoid probability map must lie in [0, 1].")

candidate_mask = probability_map > PROBABILITY_THRESHOLD
coordinates_matrix = np.column_stack(np.where(candidate_mask)).astype(np.int32)


In [ ]:
fig, axis = plt.subplots(figsize=(12, 6))
image = axis.imshow(
    probability_map.T,
    cmap="viridis",
    origin="upper",
    vmin=0.0,
    vmax=1.0,
    aspect="auto",
)
axis.set_xlim(*PLOT_X_LIMITS)
axis.set_ylim(*PLOT_Y_LIMITS)
axis.set_title("Predicted fault-probability map: inline 600", fontsize=12)
fig.colorbar(image, ax=axis, label="Fault probability")
fig.tight_layout()
plt.show()


## 2. Weak gradient-ridge support and legacy Group reduction

A strict ridge core is identified from local maxima sampled along the normalized gradient direction with a 10% section-wise gradient threshold. To preserve the source result, this core is used as spatial support, at least 99.9% of thresholded points are retained, and the original two-pass Group reduction is applied.


In [ ]:
def _disk_structure(radius):
    radius = int(radius)
    if radius < 0:
        raise ValueError('Invalid input.')
    yy, xx = np.ogrid[-radius:radius + 1, -radius:radius + 1]
    return xx * xx + yy * yy <= radius * radius


def extract_paper_ridge_core(probability_map, candidate_mask, gradient_ratio=0.1, normal_step=1.0):
    probability_map = np.asarray(probability_map, dtype=np.float32)
    candidate_mask = np.asarray(candidate_mask, dtype=bool)
    if probability_map.ndim != 2:
        raise ValueError('Invalid input.')
    if candidate_mask.shape != probability_map.shape:
        raise ValueError('Invalid input.')
    grad_axis0, grad_axis1 = np.gradient(probability_map)
    gradient_magnitude = np.hypot(grad_axis0, grad_axis1)
    max_gradient = float(gradient_magnitude.max())
    gradient_cutoff = gradient_ratio * max_gradient
    eps = np.finfo(np.float32).eps
    normal_axis0 = grad_axis0 / (gradient_magnitude + eps)
    normal_axis1 = grad_axis1 / (gradient_magnitude + eps)
    axis0, axis1 = np.indices(probability_map.shape, dtype=np.float32)
    forward_values = map_coordinates(probability_map, [axis0 + normal_step * normal_axis0, axis1 + normal_step * normal_axis1], order=1, mode='nearest')
    backward_values = map_coordinates(probability_map, [axis0 - normal_step * normal_axis0, axis1 - normal_step * normal_axis1], order=1, mode='nearest')
    local_maximum = (probability_map >= forward_values) & (probability_map >= backward_values)
    if max_gradient > 0.0:
        gradient_supported = gradient_magnitude >= gradient_cutoff
    else:
        gradient_supported = np.zeros_like(candidate_mask, dtype=bool)
    ridge_core_mask = candidate_mask & local_maximum & gradient_supported
    diagnostics = {'gradient_axis0': grad_axis0, 'gradient_axis1': grad_axis1, 'gradient_magnitude': gradient_magnitude, 'gradient_cutoff': gradient_cutoff, 'local_maximum_mask': local_maximum, 'ridge_core_mask': ridge_core_mask}
    return (ridge_core_mask, diagnostics)


def preserve_most_points_around_ridges(candidate_mask, ridge_core_mask, support_radius=8, min_keep_ratio=0.999):
    candidate_mask = np.asarray(candidate_mask, dtype=bool)
    ridge_core_mask = np.asarray(ridge_core_mask, dtype=bool)
    if candidate_mask.shape != ridge_core_mask.shape:
        raise ValueError('Invalid input.')
    if not 0.0 <= min_keep_ratio <= 1.0:
        raise ValueError('Invalid input.')
    candidate_count = int(candidate_mask.sum())
    if candidate_count == 0:
        return (candidate_mask.copy(), {'candidate_count': 0, 'ridge_core_count': 0, 'retained_count': 0, 'retention_ratio': 1.0, 'added_back_count': 0})
    if not ridge_core_mask.any() or min_keep_ratio == 1.0:
        retained_mask = candidate_mask.copy()
        return (retained_mask, {'candidate_count': candidate_count, 'ridge_core_count': int(ridge_core_mask.sum()), 'retained_count': candidate_count, 'retention_ratio': 1.0, 'added_back_count': 0})
    support_mask = binary_dilation(ridge_core_mask, structure=_disk_structure(support_radius))
    retained_mask = candidate_mask & support_mask
    target_count = int(np.ceil(min_keep_ratio * candidate_count))
    current_count = int(retained_mask.sum())
    added_back_count = 0
    if current_count < target_count:
        missing_mask = candidate_mask & ~retained_mask
        missing_coordinates = np.argwhere(missing_mask)
        distance_to_ridge = distance_transform_edt(~ridge_core_mask)
        missing_distances = distance_to_ridge[missing_mask]
        number_to_add = min(target_count - current_count, len(missing_coordinates))
        nearest_order = np.argsort(missing_distances, kind='stable')[:number_to_add]
        coordinates_to_add = missing_coordinates[nearest_order]
        retained_mask[coordinates_to_add[:, 0], coordinates_to_add[:, 1]] = True
        added_back_count = int(number_to_add)
    retained_count = int(retained_mask.sum())
    diagnostics = {'candidate_count': candidate_count, 'ridge_core_count': int(ridge_core_mask.sum()), 'retained_count': retained_count, 'retention_ratio': retained_count / candidate_count, 'added_back_count': added_back_count}
    return (retained_mask, diagnostics)


def group_by_gap_1d(values, max_gap=10):
    values = np.asarray(values)
    if values.size == 0:
        return []
    sorted_idx = np.argsort(values)
    sorted_vals = values[sorted_idx]
    groups = []
    current_group = [sorted_idx[0]]
    for i in range(1, len(sorted_vals)):
        if sorted_vals[i] - sorted_vals[i - 1] <= max_gap:
            current_group.append(sorted_idx[i])
        else:
            groups.append(current_group)
            current_group = [sorted_idx[i]]
    groups.append(current_group)
    return groups


def process_all_data(DBS_data, max_gap=10, mode='x'):
    DBS_data = np.asarray(DBS_data)
    if DBS_data.size == 0:
        return np.empty((0, 2), dtype=np.int32)
    group_f_points = []
    if mode == 'x':
        x_unique = np.unique(DBS_data[:, 0])
        for x_val in x_unique:
            ys = DBS_data[DBS_data[:, 0] == x_val, 1]
            y_groups = group_by_gap_1d(ys, max_gap=max_gap)
            for group in y_groups:
                middle = len(group) // 2
                y_representative = ys[group[middle]]
                group_f_points.append([x_val, y_representative])
    elif mode == 'y':
        y_unique = np.unique(DBS_data[:, 1])
        for y_val in y_unique:
            xs = DBS_data[DBS_data[:, 1] == y_val, 0]
            x_groups = group_by_gap_1d(xs, max_gap=max_gap)
            for group in x_groups:
                middle = len(group) // 2
                x_representative = xs[group[middle]]
                group_f_points.append([x_representative, y_val])
    else:
        raise ValueError('Invalid input.')
    return np.unique(np.asarray(group_f_points, dtype=np.int32), axis=0)


In [ ]:
ridge_core_mask, ridge_diagnostics = extract_paper_ridge_core(
    probability_map=probability_map,
    candidate_mask=candidate_mask,
    gradient_ratio=GRADIENT_RATIO,
    normal_step=NORMAL_STEP,
)
weak_ridge_mask, weak_ridge_diagnostics = preserve_most_points_around_ridges(
    candidate_mask=candidate_mask,
    ridge_core_mask=ridge_core_mask,
    support_radius=RIDGE_SUPPORT_RADIUS,
    min_keep_ratio=MIN_KEEP_RATIO,
)

ridge_core_points = np.argwhere(ridge_core_mask).astype(np.int32)
ridge_preserved_points = np.argwhere(weak_ridge_mask).astype(np.int32)

group_f_points = process_all_data(
    ridge_preserved_points,
    max_gap=GROUP_MAX_GAP,
    mode="y",
)
group_f_points = process_all_data(
    group_f_points,
    max_gap=GROUP_MAX_GAP,
    mode="x",
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
draw_points(axes[0], ridge_core_points, "Strict gradient-ridge core")
draw_points(axes[1], group_f_points, "Weak ridge support after Group reduction")
fig.tight_layout()
plt.show()


## 3. Density-based removal of isolated ridge points


In [ ]:
denoise_model = DBSCAN(
    eps=DENOISE_EPS,
    min_samples=DENOISE_MIN_SAMPLES,
).fit(group_f_points)

density_labels = denoise_model.labels_
density_mask = density_labels != -1
DBS_data = group_f_points[density_mask]
DBS_labels = density_labels[density_mask]


In [ ]:
fig, axis = plt.subplots(figsize=(12, 6))
draw_clusters(axis, DBS_data, DBS_labels, "Density-connected ridge-point groups")
fig.tight_layout()
plt.show()


## 4. Weak curvature-constrained adjustment and legacy angle filtering

Within each density-connected group, points are ordered by the dominant PCA direction, parameterized by cumulative arc length, and evaluated with the curvature-deviation ratio. Points with a ratio greater than 0.5 move 1% toward a five-point moving-average target, with displacement capped at 0.05 pixel. The source notebook's three-point angle filter is then applied unchanged.


In [ ]:
def _ensure_odd_window(window_size, n_points, minimum=3):
    if n_points < minimum:
        return None
    window_size = max(minimum, int(window_size))
    if window_size % 2 == 0:
        window_size += 1
    max_window = n_points if n_points % 2 == 1 else n_points - 1
    window_size = min(window_size, max_window)
    return window_size if window_size >= minimum else None


def _order_points_by_pca(points):
    points = np.asarray(points, dtype=np.float64)
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError('Invalid input.')
    if len(points) < 2:
        order = np.arange(len(points))
        return (order, points.copy(), np.array([1.0, 0.0]))
    centered = points - points.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    principal_direction = vh[0]
    projection = centered @ principal_direction
    order = np.argsort(projection, kind='stable')
    return (order, points[order], principal_direction)


def _arc_length_parameter(ordered_points, eps=1e-06):
    ordered_points = np.asarray(ordered_points, dtype=np.float64)
    if len(ordered_points) == 0:
        return np.empty(0, dtype=np.float64)
    if len(ordered_points) == 1:
        return np.array([0.0], dtype=np.float64)
    segment_length = np.linalg.norm(np.diff(ordered_points, axis=0), axis=1)
    segment_length = np.maximum(segment_length, eps)
    return np.concatenate([np.array([0.0], dtype=np.float64), np.cumsum(segment_length)])


def compute_paper_curvature_diagnostics(points, local_window=7, tau=0.5, delta=0.001, edge_margin=2):
    points = np.asarray(points, dtype=np.float64)
    n_points = len(points)
    empty_result = {'order': np.arange(n_points), 'ordered_points': points.copy(), 'principal_direction': np.array([1.0, 0.0]), 'arc_length': np.zeros(n_points, dtype=np.float64), 'curvature': np.zeros(n_points, dtype=np.float64), 'local_mean_curvature': np.zeros(n_points, dtype=np.float64), 'eta': np.zeros(n_points, dtype=np.float64), 'strict_anomaly_ordered': np.zeros(n_points, dtype=bool), 'strict_anomaly_original': np.zeros(n_points, dtype=bool)}
    if n_points < 5:
        return empty_result
    order, ordered_points, principal_direction = _order_points_by_pca(points)
    s = _arc_length_parameter(ordered_points)
    x = ordered_points[:, 0]
    z = ordered_points[:, 1]
    dx = np.gradient(x, s, edge_order=2)
    dz = np.gradient(z, s, edge_order=2)
    d2x = np.gradient(dx, s, edge_order=2)
    d2z = np.gradient(dz, s, edge_order=2)
    numerator = np.abs(dx * d2z - dz * d2x)
    denominator = np.power(dx * dx + dz * dz, 1.5)
    curvature = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 1e-12)
    window = _ensure_odd_window(local_window, n_points)
    if window is None:
        local_mean_curvature = curvature.copy()
    else:
        local_mean_curvature = uniform_filter1d(curvature, size=window, mode='nearest')
    eta = np.abs(curvature - local_mean_curvature) / (local_mean_curvature + float(delta))
    strict_anomaly_ordered = eta > float(tau)
    margin = min(max(int(edge_margin), 0), n_points // 2)
    if margin > 0:
        strict_anomaly_ordered[:margin] = False
        strict_anomaly_ordered[-margin:] = False
    strict_anomaly_original = np.zeros(n_points, dtype=bool)
    strict_anomaly_original[order] = strict_anomaly_ordered
    return {'order': order, 'ordered_points': ordered_points, 'principal_direction': principal_direction, 'arc_length': s, 'curvature': curvature, 'local_mean_curvature': local_mean_curvature, 'eta': eta, 'strict_anomaly_ordered': strict_anomaly_ordered, 'strict_anomaly_original': strict_anomaly_original}


def weak_paper_curvature_smoothing(points, local_window=7, smoothing_window=5, tau=0.5, delta=0.001, blend_strength=0.01, max_shift=0.05):
    points = np.asarray(points, dtype=np.float64)
    if not 0.0 <= blend_strength <= 1.0:
        raise ValueError('Invalid input.')
    if max_shift < 0.0:
        raise ValueError('Invalid input.')
    diagnostics = compute_paper_curvature_diagnostics(points, local_window=local_window, tau=tau, delta=delta)
    order = diagnostics['order']
    ordered_points = diagnostics['ordered_points']
    strict_mask = diagnostics['strict_anomaly_ordered']
    adjusted_ordered = ordered_points.copy()
    n_points = len(points)
    if n_points >= 5 and np.any(strict_mask) and (blend_strength > 0.0):
        window = _ensure_odd_window(smoothing_window, n_points)
        if window is not None:
            target = np.column_stack([uniform_filter1d(ordered_points[:, 0], size=window, mode='nearest'), uniform_filter1d(ordered_points[:, 1], size=window, mode='nearest')])
            displacement = (target - ordered_points) * float(blend_strength)
            displacement_norm = np.linalg.norm(displacement, axis=1)
            scale = np.ones(n_points, dtype=np.float64)
            too_large = displacement_norm > float(max_shift)
            scale[too_large] = float(max_shift) / displacement_norm[too_large]
            displacement *= scale[:, None]
            adjusted_ordered[strict_mask] += displacement[strict_mask]
    adjusted_points = points.copy()
    adjusted_points[order] = adjusted_ordered
    displacement_all = np.linalg.norm(adjusted_points - points, axis=1)
    diagnostics = dict(diagnostics)
    diagnostics.update({'input_count': int(n_points), 'output_count': int(n_points), 'strict_anomaly_count': int(strict_mask.sum()), 'adjusted_count': int(np.sum(displacement_all > 0.0)), 'retention_ratio': 1.0, 'mean_displacement': float(displacement_all.mean()) if n_points else 0.0, 'max_displacement': float(displacement_all.max()) if n_points else 0.0})
    return (adjusted_points, diagnostics)


def apply_weak_curvature_by_cluster(data, labels, local_window=7, smoothing_window=5, tau=0.5, delta=0.001, blend_strength=0.01, max_shift=0.05):
    data = np.asarray(data, dtype=np.float64)
    labels = np.asarray(labels)
    if len(data) != len(labels):
        raise ValueError('Invalid input.')
    adjusted_data = data.copy()
    cluster_diagnostics = {}
    for cluster_label in np.unique(labels):
        cluster_indices = np.flatnonzero(labels == cluster_label)
        cluster_points = data[cluster_indices]
        adjusted_points, diagnostics = weak_paper_curvature_smoothing(cluster_points, local_window=local_window, smoothing_window=smoothing_window, tau=tau, delta=delta, blend_strength=blend_strength, max_shift=max_shift)
        adjusted_data[cluster_indices] = adjusted_points
        cluster_diagnostics[cluster_label] = diagnostics
    total_strict = sum((d['strict_anomaly_count'] for d in cluster_diagnostics.values()))
    total_adjusted = sum((d['adjusted_count'] for d in cluster_diagnostics.values()))
    all_displacements = np.linalg.norm(adjusted_data - data, axis=1)
    summary = {'input_count': int(len(data)), 'output_count': int(len(adjusted_data)), 'retention_ratio': 1.0, 'strict_anomaly_count': int(total_strict), 'adjusted_count': int(total_adjusted), 'mean_displacement': float(all_displacements.mean()) if len(data) else 0.0, 'max_displacement': float(all_displacements.max()) if len(data) else 0.0, 'cluster_diagnostics': cluster_diagnostics}
    return (adjusted_data, labels.copy(), summary)


def compute_angle(p1, p2, p3):
    v1 = np.asarray(p1, dtype=np.float64) - np.asarray(p2, dtype=np.float64)
    v2 = np.asarray(p3, dtype=np.float64) - np.asarray(p2, dtype=np.float64)
    denominator = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denominator == 0.0:
        return 180.0
    cosine = np.clip(np.dot(v1, v2) / denominator, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def find_bad_indices_by_sorted_angle(data, axis, reverse=False, angle_threshold=90.0):
    data = np.asarray(data, dtype=np.float64)
    if len(data) < 3:
        return set()
    sorted_idx = np.argsort(data[:, axis], kind='stable')[::-1] if reverse else np.argsort(data[:, axis], kind='stable')
    sorted_data = data[sorted_idx]
    bad_indices = set()
    for i in range(1, len(sorted_data) - 1):
        p1 = sorted_data[i - 1]
        p2 = sorted_data[i]
        p3 = sorted_data[i + 1]
        angle = compute_angle(p1, p2, p3)
        if angle < float(angle_threshold):
            bad_indices.add(int(sorted_idx[i]))
    return bad_indices


def apply_original_angle_filter(data, labels, angle_threshold=90.0, max_rounds=10):
    data = np.asarray(data, dtype=np.float64)
    labels = np.asarray(labels)
    if len(data) != len(labels):
        raise ValueError('Invalid input.')
    cleaned_data_list = []
    cleaned_label_list = []
    removed_by_cluster = {}
    for cluster_label in np.unique(labels):
        cluster_mask = labels == cluster_label
        cleaned = data[cluster_mask].copy()
        cleaned_labels = labels[cluster_mask].copy()
        original_count = len(cleaned)
        for _ in range(int(max_rounds)):
            bad_idx_set = set()
            bad_idx_set |= find_bad_indices_by_sorted_angle(cleaned, axis=0, reverse=False, angle_threshold=angle_threshold)
            bad_idx_set |= find_bad_indices_by_sorted_angle(cleaned, axis=0, reverse=True, angle_threshold=angle_threshold)
            bad_idx_set |= find_bad_indices_by_sorted_angle(cleaned, axis=1, reverse=False, angle_threshold=angle_threshold)
            bad_idx_set |= find_bad_indices_by_sorted_angle(cleaned, axis=1, reverse=True, angle_threshold=angle_threshold)
            if not bad_idx_set:
                break
            keep_mask = np.array([i not in bad_idx_set for i in range(len(cleaned))], dtype=bool)
            cleaned = cleaned[keep_mask]
            cleaned_labels = cleaned_labels[keep_mask]
            if len(cleaned) < 3:
                break
        removed_by_cluster[cluster_label] = int(original_count - len(cleaned))
        if len(cleaned):
            cleaned_data_list.append(cleaned)
            cleaned_label_list.append(cleaned_labels)
    if cleaned_data_list:
        cleaned_data = np.vstack(cleaned_data_list)
        cleaned_labels = np.concatenate(cleaned_label_list)
    else:
        cleaned_data = np.empty((0, 2), dtype=np.float64)
        cleaned_labels = np.empty(0, dtype=labels.dtype)
    summary = {'input_count': int(len(data)), 'output_count': int(len(cleaned_data)), 'removed_count': int(len(data) - len(cleaned_data)), 'retention_ratio': len(cleaned_data) / len(data) if len(data) else 1.0, 'removed_by_cluster': removed_by_cluster}
    return (cleaned_data, cleaned_labels, summary)


In [ ]:
curvature_data, curvature_labels, curvature_summary = apply_weak_curvature_by_cluster(
    data=DBS_data,
    labels=DBS_labels,
    local_window=CURVATURE_LOCAL_WINDOW,
    smoothing_window=CURVATURE_SMOOTH_WINDOW,
    tau=CURVATURE_TAU,
    delta=CURVATURE_DELTA,
    blend_strength=CURVATURE_BLEND_STRENGTH,
    max_shift=CURVATURE_MAX_SHIFT,
)

break_data, break_labels, angle_summary = apply_original_angle_filter(
    data=curvature_data,
    labels=curvature_labels,
    angle_threshold=ANGLE_THRESHOLD,
    max_rounds=ANGLE_MAX_ROUNDS,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
draw_clusters(axes[0], curvature_data, curvature_labels, "Weak curvature adjustment")
draw_clusters(axes[1], break_data, break_labels, "After legacy three-point angle filtering")
fig.tight_layout()
plt.show()


## 5. Second DBSCAN noise mask

This cell deliberately preserves the source notebook's behavior: the second DBSCAN identifies its noise mask, while the surviving points retain the labels carried from the first DBSCAN stage.


In [ ]:
second_dbscan = DBSCAN(
    eps=SECOND_DBSCAN_EPS,
    min_samples=SECOND_DBSCAN_MIN_SAMPLES,
).fit(break_data)
second_dbscan_labels = second_dbscan.labels_
second_density_mask = second_dbscan_labels != -1
DBS_data2 = break_data[second_density_mask]
DBS_labels2 = break_labels[second_density_mask]


In [ ]:
fig, axis = plt.subplots(figsize=(12, 6))
draw_clusters(axis, DBS_data2, DBS_labels2, "Clusters entering geometry-based merging")
fig.tight_layout()
plt.show()


## 6. Iterative geometry-consistent cluster merging

Every current cluster is represented by its centroid and PCA axis. A pair is merged only when its axial-angle difference, all-point minimum distance, and directional mean extrapolation distance simultaneously satisfy the selected thresholds. Cluster geometry is recomputed after every batch of merges until convergence.

The manuscript reference values are 15 degrees, 75 pixels, and 5 pixels. The source notebook uses the calibrated in-range values 17 degrees, 75 pixels, and 8 pixels to preserve its legacy output.


In [ ]:
def _pca_cluster_geometry(points):
    points = np.asarray(points, dtype=float)
    if points.ndim != 2 or points.shape[1] != 2 or len(points) == 0:
        raise ValueError('Each cluster must contain at least one 2D point.')
    centroid = points.mean(axis=0)
    centered = points - centroid
    if len(points) == 1 or np.allclose(centered, 0.0):
        direction = np.array([1.0, 0.0])
    else:
        _, _, vh = np.linalg.svd(centered, full_matrices=False)
        direction = vh[0]
        direction = direction / np.linalg.norm(direction)
    return {'centroid': centroid, 'direction': direction, 'tree': cKDTree(points)}


def _mean_distance_to_pca_line(points, centroid, direction):
    offsets = np.asarray(points, dtype=float) - centroid
    along_line = np.outer(offsets @ direction, direction)
    orthogonal = offsets - along_line
    return float(np.mean(np.linalg.norm(orthogonal, axis=1)))


def _paper_pair_metrics(points_i, geometry_i, points_j, geometry_j):
    cosine = np.clip(abs(np.dot(geometry_i['direction'], geometry_j['direction'])), 0.0, 1.0)
    theta_ij = float(np.degrees(np.arccos(cosine)))
    d_ij = float(np.min(geometry_i['tree'].query(points_j, k=1)[0]))
    e_ij = _mean_distance_to_pca_line(points_j, geometry_i['centroid'], geometry_i['direction'])
    e_ji = _mean_distance_to_pca_line(points_i, geometry_j['centroid'], geometry_j['direction'])
    return {'theta': theta_ij, 'distance': d_ij, 'extrapolation': min(e_ij, e_ji), 'extrapolation_i_to_j': e_ij, 'extrapolation_j_to_i': e_ji}


def merge_clusters_by_paper_geometry(data, labels, theta_max, distance_max, extrapolation_max):
    data = np.asarray(data, dtype=float)
    labels = np.asarray(labels)
    if data.ndim != 2 or data.shape[1] != 2:
        raise ValueError('data must be an N×2 array.')
    if len(data) != len(labels):
        raise ValueError('data and labels must have the same length.')
    if theta_max <= 0 or distance_max <= 0 or extrapolation_max <= 0:
        raise ValueError('All merge thresholds must be positive.')
    original_labels = np.unique(labels)
    if len(original_labels) == 0:
        summary = {'input_clusters': 0, 'final_clusters': 0, 'parameters': {'theta_max': theta_max, 'distance_max': distance_max, 'extrapolation_max': extrapolation_max}, 'iterations': []}
        return (np.empty(0, dtype=int), summary)
    clusters = {label: {'points': data[labels == label], 'members': {label}} for label in original_labels}
    history = []
    for iteration in range(1, len(original_labels) + 1):
        cluster_ids = sorted(clusters)
        geometries = {cluster_id: _pca_cluster_geometry(clusters[cluster_id]['points']) for cluster_id in cluster_ids}
        parent = {cluster_id: cluster_id for cluster_id in cluster_ids}

        def find(cluster_id):
            while parent[cluster_id] != cluster_id:
                parent[cluster_id] = parent[parent[cluster_id]]
                cluster_id = parent[cluster_id]
            return cluster_id

        def union(cluster_i, cluster_j):
            root_i = find(cluster_i)
            root_j = find(cluster_j)
            if root_i != root_j:
                parent[root_j] = root_i
        qualifying_pairs = 0
        for position, cluster_i in enumerate(cluster_ids):
            points_i = clusters[cluster_i]['points']
            for cluster_j in cluster_ids[position + 1:]:
                points_j = clusters[cluster_j]['points']
                metrics = _paper_pair_metrics(points_i, geometries[cluster_i], points_j, geometries[cluster_j])
                if metrics['theta'] < theta_max and metrics['distance'] < distance_max and (metrics['extrapolation'] < extrapolation_max):
                    union(cluster_i, cluster_j)
                    qualifying_pairs += 1
        components = defaultdict(list)
        for cluster_id in cluster_ids:
            components[find(cluster_id)].append(cluster_id)
        output_cluster_count = len(components)
        history.append({'iteration': iteration, 'input_clusters': len(cluster_ids), 'candidate_pairs': len(cluster_ids) * (len(cluster_ids) - 1) // 2, 'qualifying_pairs': qualifying_pairs, 'output_clusters': output_cluster_count})
        if output_cluster_count == len(cluster_ids):
            break
        recomputed_clusters = {}
        for component_ids in components.values():
            component_id = min(component_ids)
            recomputed_clusters[component_id] = {'points': np.vstack([clusters[cluster_id]['points'] for cluster_id in component_ids]), 'members': set().union(*[clusters[cluster_id]['members'] for cluster_id in component_ids])}
        clusters = recomputed_clusters
    else:
        raise RuntimeError('Iterative cluster merging did not converge.')
    original_to_merged = {}
    ordered_clusters = sorted(clusters.values(), key=lambda cluster: min(cluster['members']))
    for merged_label, cluster in enumerate(ordered_clusters):
        for original_label in cluster['members']:
            original_to_merged[original_label] = merged_label
    merged_labels = np.asarray([original_to_merged[label] for label in labels], dtype=int)
    summary = {'input_clusters': int(len(original_labels)), 'final_clusters': int(len(ordered_clusters)), 'parameters': {'theta_max': float(theta_max), 'distance_max': float(distance_max), 'extrapolation_max': float(extrapolation_max)}, 'iterations': history}
    return (merged_labels, summary)


In [ ]:
merged_labels, paper_merge_summary = merge_clusters_by_paper_geometry(
    data=DBS_data2,
    labels=DBS_labels2,
    theta_max=MERGE_THETA_MAX,
    distance_max=MERGE_DISTANCE_MAX,
    extrapolation_max=MERGE_EXTRAPOLATION_MAX,
)
unique_merged_labels = np.unique(merged_labels)
n_merged_clusters = len(unique_merged_labels)


In [ ]:
fig, axis = plt.subplots(figsize=(12, 6))
draw_clusters(axis, DBS_data2, merged_labels, "Iteratively merged fault-line clusters")
fig.tight_layout()
plt.show()


## 7. Large-cluster selection and repeated geometry cleanup

The source notebook retains clusters with at least 30 points and a vertical range greater than 30 pixels. The same weak curvature adjustment and three-point angle filter are then applied a second time without changing their parameters.


In [ ]:
merged_clusters = defaultdict(list)
for point, cluster_label in zip(DBS_data2, merged_labels):
    merged_clusters[cluster_label].append(point)
for cluster_label in merged_clusters:
    merged_clusters[cluster_label] = np.asarray(merged_clusters[cluster_label])

big_clusters = {}
for cluster_points in merged_clusters.values():
    vertical_range = np.max(cluster_points[:, 1]) - np.min(cluster_points[:, 1])
    if len(cluster_points) >= MIN_CLUSTER_POINTS and vertical_range > MIN_CLUSTER_VERTICAL_RANGE:
        big_clusters[len(big_clusters)] = cluster_points

big_data = np.vstack(list(big_clusters.values()))
big_labels = np.concatenate([
    np.full(len(cluster_points), cluster_label)
    for cluster_label, cluster_points in big_clusters.items()
])
unique_big_labels = np.unique(big_labels)

big_curvature_data, big_curvature_labels, big_curvature_summary = apply_weak_curvature_by_cluster(
    data=big_data,
    labels=big_labels,
    local_window=CURVATURE_LOCAL_WINDOW,
    smoothing_window=CURVATURE_SMOOTH_WINDOW,
    tau=CURVATURE_TAU,
    delta=CURVATURE_DELTA,
    blend_strength=CURVATURE_BLEND_STRENGTH,
    max_shift=CURVATURE_MAX_SHIFT,
)

break_data2, break_labels2, angle_summary2 = apply_original_angle_filter(
    data=big_curvature_data,
    labels=big_curvature_labels,
    angle_threshold=ANGLE_THRESHOLD,
    max_rounds=ANGLE_MAX_ROUNDS,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
draw_clusters(axes[0], big_data, big_labels, "Selected large clusters")
draw_clusters(axes[1], break_data2, break_labels2, "Repeated curvature and angle cleanup")
fig.tight_layout()
plt.show()


## 8. Linear interpolation of the final 2D fault traces


In [ ]:
interpolated_clusters = {}
for cluster_label in unique_big_labels:
    cluster_points = break_data2[break_labels2 == cluster_label]
    cluster_points = cluster_points[np.argsort(cluster_points[:, 0])]
    x_coordinates = cluster_points[:, 0]
    y_coordinates = cluster_points[:, 1]

    interpolation = interp1d(
        x_coordinates,
        y_coordinates,
        kind="linear",
        fill_value="extrapolate",
    )
    x_new = np.arange(
        np.min(x_coordinates),
        np.max(x_coordinates) + 1,
    ).astype(int)
    y_new = np.round(interpolation(x_new)).astype(int)
    interpolated_clusters[cluster_label] = np.stack((x_new, y_new), axis=1)

interpolated_points_numpy = np.concatenate(list(interpolated_clusters.values()), axis=0)
interpolated_labels_numpy = np.concatenate([
    np.full(len(cluster_points), cluster_label)
    for cluster_label, cluster_points in interpolated_clusters.items()
])

# Silent regression checks for the inline-600 reference result.
assert probability_map.shape == EXPECTED_PROBABILITY_SHAPE
assert probability_map.shape == seismic_normalized.shape
assert probability_map.dtype == np.float32
assert np.isfinite(probability_map).all()
assert 0.0 <= float(probability_map.min()) <= float(probability_map.max()) <= 1.0
assert len(coordinates_matrix) == 32195
assert len(ridge_core_points) == 836
assert len(ridge_preserved_points) == 32163
assert len(group_f_points) == 2253
assert len(DBS_data) == 2031
assert np.unique(DBS_labels).size == 114
assert len(curvature_data) == 2031
assert len(break_data) == 1968
assert len(DBS_data2) == 1967
assert np.unique(merged_labels).size == 81
assert len(big_data) == 1254
assert np.unique(big_labels).size == 15
assert len(break_data2) == 1202
assert interpolated_points_numpy.shape == (1927, 2)
assert interpolated_labels_numpy.shape == (1927,)
assert np.unique(interpolated_labels_numpy).size == 15


In [ ]:
fig, axis = plt.subplots(figsize=(12, 6))
draw_clusters(
    axis,
    interpolated_points_numpy,
    interpolated_labels_numpy,
    "Final interpolated 2D fault traces",
)
fig.tight_layout()
plt.show()


## 9. Final fault traces overlaid on the inference seismic section

The background below is the same normalized inline-600 section used to generate the UASS-Net probability map.

In [ ]:
if seismic_normalized.shape != probability_map.shape:
    raise ValueError(
        "The seismic section and predicted probability map must have the same shape."
    )


In [ ]:
fig, axis = plt.subplots(figsize=(12, 6))
axis.imshow(
    seismic_normalized.T,
    cmap="gray",
    vmin=0.44,
    vmax=0.54,
    origin="upper",
    aspect="auto",
)
colors = [
    plt.cm.Spectral(value)
    for value in np.linspace(0, 1, len(interpolated_clusters))
]
for (cluster_label, cluster_points), color in zip(interpolated_clusters.items(), colors):
    axis.plot(
        cluster_points[:, 0],
        cluster_points[:, 1],
        "-",
        color=color,
        linewidth=2,
    )
axis.invert_xaxis()
axis.set_title("Final reconstructed fault traces: inline 600", fontsize=12)
axis.tick_params(labelsize=11)
fig.tight_layout()
plt.show()
